### Cleaning ideology data and building senator_ideologies_df

In [75]:
# imports
import pandas as pd

In [76]:
# load ideology score files
nominate = pd.read_csv('../../data/ideology_scores/nominate.csv')
govtrack  = pd.read_csv('../../data/ideology_scores/govtrack.csv')

---
### NOMINATE

Key columns:
- `congress` — which congress number (1 = 1789, 119 = current)
- `chamber` — Senate, House, or President
- `bioguide_id` — matches the senators dataframe, main merge key
- `born` / `died` — birth and death years
- `nominate_dim1` — main ideology score: negative = liberal, positive = conservative
- `nominate_dim2` — secondary dimension, historically captured cross-cutting issues like race/civil rights
- `nominate_log_likelihood` — model fit stat, how well votes fit the estimated position
- `nominate_geo_mean_probability` — another fit stat, higher = more predictable votes
- `nominate_number_of_votes` — how many votes cast that congress
- `nominate_number_of_errors` — how many votes didn't match predicted position
- `nokken_poole_dim1/2` — alternative score that allows positions to shift congress-to-congress

In [77]:
# drop rows with no bioguide_id or no scores
nominate = nominate.dropna(subset=['bioguide_id', 'nominate_dim1'])

# keep only senate rows
nominate = nominate[nominate['chamber'] == 'Senate']

# parse names before aggregating
def parse_bioname(name):
    if not isinstance(name, str):
        return None, None
    parts = name.split(',', 1)
    last  = parts[0].strip().title()
    first = parts[1].strip().title() if len(parts) > 1 else None
    return first, last

nominate[['first_name', 'last_name']] = nominate['bioname'].apply(
    lambda x: pd.Series(parse_bioname(x))
)
nominate = nominate.drop(columns=['bioname'])

In [78]:
# NOW aggregate — one row per senator
nominate_clean = nominate.groupby('bioguide_id').agg(
    nominate_dim1=('nominate_dim1', 'mean'),
    nominate_dim2=('nominate_dim2', 'mean')
).reset_index()

print(nominate_clean.shape)
print(nominate_clean.columns.tolist())

(1961, 3)
['bioguide_id', 'nominate_dim1', 'nominate_dim2']


---
### GovTrack

Key columns:
- `ideology` — GovTrack ideology score, 0.0 (most liberal) to 1.0 (most conservative)
- `bioguide_id` — merge key
- `rank_from_low` — rank from most liberal (1) to most conservative
- `rank_from_high` — flip side of rank_from_low
- `percentile` — percentile from the liberal end
- `id` — GovTrack internal numeric ID, not useful for merging

In [79]:
# clean the name col — strip the b'' bytes literal formatting
govtrack['name'] = govtrack['name'].str.replace(r"^b'|'$", "", regex=True)

# only need these cols
govtrack_clean = govtrack[['bioguide_id', 'ideology']].rename(
    columns={'ideology': 'govtrack_ideology_score'}
)

In [80]:
govtrack_clean.head()

,bioguide_id,govtrack_ideology_score
0,S000033,0.000000
1,W000800,0.056155
2,M001176,0.072088
3,B001277,0.083912
4,W000817,0.083938


---
### Save cleaned files

In [81]:
nominate_clean.to_csv('../../data/ideology_scores/nominate_cleaned.csv', index=False)
govtrack_clean.to_csv('../../data/ideology_scores/govtrack_cleaned.csv', index=False)

---
### Build senator_ideologies_df

Merge nominate_clean and govtrack_clean on bioguide_id.
- Use an outer join so senators who appear in one source but not the other are still included.
- nominate_clean is the left frame since it covers a much longer historical range.

In [82]:
# merge dfs
senator_ideologies_df = nominate_clean.merge(
    govtrack_clean,
    on='bioguide_id',
    how='outer'
)

print(f'rows: {senator_ideologies_df.shape[0]}')
print(f'columns: {senator_ideologies_df.columns.tolist()}')
senator_ideologies_df.head()

rows: 1961
columns: ['bioguide_id', 'nominate_dim1', 'nominate_dim2', 'govtrack_ideology_score']


,bioguide_id,nominate_dim1,nominate_dim2,govtrack_ideology_score
0,A000006,0.290,0.836,NaN
1,A000009,0.241,0.129,NaN
2,A000017,-0.743,0.423,NaN
3,A000026,-0.251,-0.051,NaN
4,A000028,-0.064,0.228,NaN


In [83]:
# save combined cleaned file
senator_ideologies_df.to_csv('../../data/ideology_scores/senator_ideologies.csv', index=False)

## What these values mean:

#### nominate_dim1 
is the main left-right ideology score. Negative values are liberal, positive values are conservative. This is the one that maps cleanly onto the familiar Democrat-Republican spectrum. It's the most important and widely used of the two — when people cite "DW-NOMINATE scores" in political science research, they're almost always talking about dim1.

#### nominate_dim2 
historically captured a second axis of political disagreement that was separate from the main left-right divide. For most of American history this second dimension tracked issues around race and civil rights — during the mid-20th century it separated Northern Democrats (liberal on civil rights) from Southern Democrats (conservative on civil rights), even though both groups were similarly liberal on economic issues. In the modern Congress (roughly post-1980s) this dimension has collapsed almost to noise because the parties have sorted so cleanly along a single axis. For senators from the last few decades, dim2 will be near zero for almost everyone and won't add much signal.

#### govtrack_ideology_score
GovTrack's ideology score measures how far left or right a senator is based on which bills they sponsor and cosponsor — not their votes. It runs from 0.0 (most liberal) to 1.0 (most conservative). The key difference from NOMINATE is the data source: NOMINATE is built from voting records, GovTrack is built from sponsorship patterns. In practice they tend to agree closely, but they can diverge for senators who vote with their party but independently choose to sponsor more cross-aisle legislation (or vice versa). Having both gives you a more complete picture of ideology from two different behavioral angles.